# 06 — Step 3: Synthetic Background Training & Cross-Background Robustness

**Project:** Tomato Quality Semantic Segmentation — Background Bias Analysis
**Models:** U-Net with MobileNetV2 & EfficientNet-B0 Encoders

## What this notebook does

1. Loads fixed splits, class weights, and Step 1 & 2 results
2. Visualises synthetic background compositing on sample images
3. Trains U-Net/MobileNetV2 on synthetic-background images
4. Trains U-Net/EfficientNet-B0 on synthetic-background images
5. Evaluates both models on the held-out test set (once only, synthetic BG)
6. Plots training curves, confusion matrices, per-class IoU, calibration
7. Generates prediction visualisations and Grad-CAM maps
8. **Cross-background robustness evaluation** — three transfer scenarios:
   - (a) Natural-trained → tested on Synthetic test set
   - (b) Synthetic-trained → tested on Natural test set
   - (c) Synthetic-trained → tested on Synthetic test set (matched)
9. **Steps 1–2–3 comparison** — metrics table, ECE summary, per-class IoU heatmap
10. Saves `step3_results.pkl` and `robustness_results.pkl` for notebook 07

> **Research question:** Do models trained on synthetic backgrounds generalise
> better across background conditions? Which encoder is more robust to background shift?

## 1. Imports

In [ ]:
import sys
import json
import time
import copy
import random
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from tqdm import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi']    = 100

print('Imports OK')

## 2. Load Shared Config

In [ ]:
sys.path.insert(0, str(Path('..').resolve()))
from config import *

print(f'Device     : {DEVICE}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

## 3. Load Splits, Class Weights & Prior Step Results

In [ ]:
# ── Splits ───────────────────────────────────────────────────────────────────
assert SPLIT_FILE.exists(), f'Run notebook 01 first: {SPLIT_FILE}'
with open(SPLIT_FILE) as f:
    split_data = json.load(f)

train_pairs = [(INPUT_DIR / img, INPUT_DIR / ann) for img, ann in split_data['train']]
val_pairs   = [(INPUT_DIR / img, INPUT_DIR / ann) for img, ann in split_data['val']]
test_pairs  = [(INPUT_DIR / img, INPUT_DIR / ann) for img, ann in split_data['test']]
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

# ── Class weights ────────────────────────────────────────────────────────────
WEIGHTS_FILE = OUTPUT_DIR / 'class_weights.npy'
assert WEIGHTS_FILE.exists(), f'Run notebook 02 first: {WEIGHTS_FILE}'
class_weights        = np.load(WEIGHTS_FILE)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'Class weights: {class_weights_tensor}')

# ── Synthetic background paths ────────────────────────────────────────────────
BG_PATHS_FILE = OUTPUT_DIR / 'synthetic_bg_paths.json'
assert BG_PATHS_FILE.exists(), f'Run notebook 02 first: {BG_PATHS_FILE}'
with open(BG_PATHS_FILE) as f:
    synthetic_bg_paths = json.load(f)
print(f'Synthetic backgrounds loaded: {len(synthetic_bg_paths)}')

# ── Step 1 results ────────────────────────────────────────────────────────────
STEP1_FILE = OUTPUT_DIR / 'step1_results.pkl'
assert STEP1_FILE.exists(), f'Run notebook 04 first: {STEP1_FILE}'
with open(STEP1_FILE, 'rb') as f:
    step1_results = pickle.load(f)

metrics_m_natural = step1_results['mobilenet']['metrics']
metrics_e_natural = step1_results['efficientnet']['metrics']
cal_m_natural     = step1_results['mobilenet']['calibration']
cal_e_natural     = step1_results['efficientnet']['calibration']

# ── Step 2 results ────────────────────────────────────────────────────────────
STEP2_FILE = OUTPUT_DIR / 'step2_results.pkl'
assert STEP2_FILE.exists(), f'Run notebook 05 first: {STEP2_FILE}'
with open(STEP2_FILE, 'rb') as f:
    step2_results = pickle.load(f)

metrics_m_removed = step2_results['mobilenet']['metrics']
metrics_e_removed = step2_results['efficientnet']['metrics']
cal_m_removed     = step2_results['mobilenet']['calibration']
cal_e_removed     = step2_results['efficientnet']['calibration']

print('Step 1 & 2 results loaded for comparison.')

## 4. Data Pipeline Functions

In [ ]:
# Self-contained copy — identical to notebooks 02, 04, 05.

def generate_mask_from_annotation(ann_path, img_height=None, img_width=None):
    with open(ann_path, 'r') as f:
        ann = json.load(f)
    h = img_height or ann['size']['height']
    w = img_width  or ann['size']['width']
    mask = np.zeros((h, w), dtype=np.uint8)
    objects_by_class = defaultdict(list)
    for obj in ann['objects']:
        ct = obj.get('classTitle', '')
        if ct in CLASS_TITLE_TO_IDX:
            objects_by_class[ct].append(obj)
    for cls_title in PAINT_PRIORITY:
        if cls_title not in objects_by_class:
            continue
        cls_idx = CLASS_TITLE_TO_IDX[cls_title]
        for obj in objects_by_class[cls_title]:
            pts = np.array(obj['points']['exterior'], dtype=np.int32)
            cv2.fillPoly(mask, [pts], cls_idx)
    return mask


def get_foreground_binary_mask(ann_path, img_height=None, img_width=None):
    with open(ann_path, 'r') as f:
        ann = json.load(f)
    h = img_height or ann['size']['height']
    w = img_width  or ann['size']['width']
    fg = np.zeros((h, w), dtype=np.uint8)
    for obj in ann['objects']:
        if obj.get('classTitle', '') in CLASS_TITLE_TO_IDX:
            pts = np.array(obj['points']['exterior'], dtype=np.int32)
            cv2.fillPoly(fg, [pts], 1)
    return fg


def get_train_augmentation(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_augmentation(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


get_test_augmentation = get_val_augmentation


class TomatoSegDataset(Dataset):
    def __init__(self, pairs, transform=None, bg_mode='natural', bg_images=None):
        self.pairs     = pairs
        self.transform = transform
        self.bg_mode   = bg_mode
        self.bg_images = bg_images or []
        missing = [(str(i), str(a)) for i, a in pairs if not i.exists() or not a.exists()]
        if missing:
            raise FileNotFoundError(f'{len(missing)} missing files. First: {missing[0]}')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, ann_p = self.pairs[idx]
        img = cv2.imread(str(img_p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = generate_mask_from_annotation(ann_p, h, w)
        if self.bg_mode == 'removed':
            fg  = get_foreground_binary_mask(ann_p, h, w)
            bg  = np.full_like(img, NEUTRAL_BG_COLOR, dtype=np.uint8)
            fg3 = np.stack([fg] * 3, axis=-1)
            img = np.where(fg3 == 1, img, bg)
        elif self.bg_mode == 'synthetic' and len(self.bg_images) > 0:
            fg     = get_foreground_binary_mask(ann_p, h, w)
            bg_img = cv2.imread(random.choice(self.bg_images))
            if bg_img is not None:
                bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
                bg_img = cv2.resize(bg_img, (w, h))
            else:
                bg_img = np.random.randint(0, 255, img.shape, dtype=np.uint8)
            fg3 = np.stack([fg] * 3, axis=-1)
            img = np.where(fg3 == 1, img, bg_img)
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask']
        return img, mask.long()


def make_loaders(train_pairs, val_pairs, test_pairs, bg_mode, bg_images=None):
    train_ds = TomatoSegDataset(train_pairs, transform=get_train_augmentation(),
                                bg_mode=bg_mode, bg_images=bg_images)
    val_ds   = TomatoSegDataset(val_pairs,   transform=get_val_augmentation(),
                                bg_mode=bg_mode, bg_images=bg_images)
    test_ds  = TomatoSegDataset(test_pairs,  transform=get_test_augmentation(),
                                bg_mode=bg_mode, bg_images=bg_images)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE_VAL,   shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE_TEST,  shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


print('Data pipeline functions defined.')

## 5. Model, Loss & Metric Utilities

In [ ]:
def create_unet_model(encoder_name, num_classes=NUM_CLASSES, pretrained='imagenet'):
    return smp.Unet(encoder_name=encoder_name, encoder_weights=pretrained,
                    in_channels=3, classes=num_classes, activation=None).to(DEVICE)


class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None, dice_weight=0.5, ce_weight=0.5,
                 num_classes=NUM_CLASSES, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.num_classes = num_classes
        self.smooth      = smooth
        self.ce_loss     = nn.CrossEntropyLoss(weight=class_weights) \
            if class_weights is not None else nn.CrossEntropyLoss()

    def dice_loss(self, pred, target):
        pred_soft      = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        intersection   = (pred_soft * target_one_hot).sum(dim=(2, 3))
        cardinality    = pred_soft.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
        return 1.0 - ((2.0 * intersection + self.smooth) / (cardinality + self.smooth)).mean()

    def forward(self, pred, target):
        return (self.ce_weight * self.ce_loss(pred, target) +
                self.dice_weight * self.dice_loss(pred, target))


def compute_metrics(pred_masks, gt_masks, num_classes=NUM_CLASSES):
    if not pred_masks or not gt_masks:
        return {'pixel_accuracy': 0., 'mean_iou': 0., 'mean_dice': 0.,
                'iou_per_class': np.zeros(num_classes), 'dice_per_class': np.zeros(num_classes),
                'acc_per_class': np.zeros(num_classes),
                'confusion_matrix': np.zeros((num_classes, num_classes)),
                'class_present': np.zeros(num_classes, dtype=bool)}
    all_pred  = np.concatenate([m.flatten() for m in pred_masks])
    all_gt    = np.concatenate([m.flatten() for m in gt_masks])
    pixel_acc = (all_pred == all_gt).mean()
    iou_pc = np.zeros(num_classes); dice_pc = np.zeros(num_classes)
    acc_pc = np.zeros(num_classes); present = np.zeros(num_classes, dtype=bool)
    for c in range(num_classes):
        pc = (all_pred == c); gc = (all_gt == c)
        ps = pc.sum();        gs = gc.sum()
        if gs == 0 and ps == 0:
            iou_pc[c] = dice_pc[c] = acc_pc[c] = float('nan'); continue
        present[c] = True
        inter = (pc & gc).sum(); union = (pc | gc).sum()
        iou_pc[c]  = inter / union         if union > 0   else 0.
        dice_pc[c] = 2 * inter / (ps + gs) if ps + gs > 0 else 0.
        acc_pc[c]  = inter / gs            if gs > 0      else 0.
    mean_iou  = np.nanmean(iou_pc [present]) if present.any() else 0.
    mean_dice = np.nanmean(dice_pc[present]) if present.any() else 0.
    cm = sk_confusion_matrix(all_gt, all_pred, labels=list(range(num_classes)))
    return {'pixel_accuracy': pixel_acc, 'mean_iou': mean_iou, 'mean_dice': mean_dice,
            'iou_per_class': iou_pc, 'dice_per_class': dice_pc, 'acc_per_class': acc_pc,
            'confusion_matrix': cm, 'class_present': present}


def print_metrics(metrics, title='Evaluation Results'):
    print(f"\n{'='*62}\n  {title}\n{'='*62}")
    print(f"  Pixel Accuracy : {metrics['pixel_accuracy']:.4f}")
    print(f"  Mean IoU       : {metrics['mean_iou']:.4f}")
    print(f"  Mean Dice      : {metrics['mean_dice']:.4f}")
    print(f"\n  {'Class':<20} {'IoU':>8} {'Dice':>8} {'Acc':>8}")
    print(f"  {'-'*46}")
    for c in range(NUM_CLASSES):
        i = metrics['iou_per_class'][c]
        d = metrics['dice_per_class'][c]
        a = metrics['acc_per_class'][c]
        print(f"  {CLASS_NAMES[c]:<20} "
              f"{'N/A' if np.isnan(i) else f'{i:.4f}':>8} "
              f"{'N/A' if np.isnan(d) else f'{d:.4f}':>8} "
              f"{'N/A' if np.isnan(a) else f'{a:.4f}':>8}")
    print(f"{'='*62}")


def compute_calibration(pred_probs_list, gt_masks_list, num_bins=15):
    all_conf = []; all_correct = []
    for probs, gt in zip(pred_probs_list, gt_masks_list):
        all_conf.append(probs.max(axis=-1).flatten())
        all_correct.append((probs.argmax(axis=-1) == gt).astype(np.float32).flatten())
    all_conf    = np.concatenate(all_conf)
    all_correct = np.concatenate(all_correct)
    bins = np.linspace(0, 1, num_bins + 1)
    bc = np.zeros(num_bins); ba = np.zeros(num_bins); bct = np.zeros(num_bins)
    for i in range(num_bins):
        m = (all_conf > bins[i]) & (all_conf <= bins[i + 1])
        if m.sum() > 0:
            bc[i] = all_conf[m].mean(); ba[i] = all_correct[m].mean(); bct[i] = m.sum()
    ece = np.sum(bct / bct.sum() * np.abs(ba - bc))
    return {'ece': ece, 'bin_confidences': bc, 'bin_accuracies': ba,
            'bin_counts': bct, 'bin_boundaries': bins}


def plot_reliability_diagram(cal_data, title='Reliability Diagram', ax=None):
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(7, 7))
    m = cal_data['bin_counts'] > 0
    ax.bar(cal_data['bin_confidences'][m], cal_data['bin_accuracies'][m],
           width=1. / len(cal_data['bin_counts']), alpha=0.6,
           color='steelblue', edgecolor='navy', label='Actual accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    ax.set_xlabel('Confidence', fontsize=12); ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'{title}\nECE = {cal_data["ece"]:.4f}', fontsize=13)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=11); ax.set_aspect('equal')
    return ax


print('Model, loss, metric utilities defined.')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device=DEVICE):
    model.train()
    rl   = 0.
    pbar = tqdm(loader, desc='  Train', leave=False, unit='batch')
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        rl += loss.item() * imgs.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return rl / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device=DEVICE, return_predictions=False):
    model.eval()
    rl = 0.; pm = []; gm = []; pp = []
    pbar = tqdm(loader, desc='  Eval ', leave=False, unit='batch')
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        out  = model(imgs)
        rl  += criterion(out, masks).item() * imgs.size(0)
        probs = F.softmax(out, dim=1)
        preds = probs.argmax(dim=1)
        for b in range(preds.shape[0]):
            pm.append(preds[b].cpu().numpy())
            gm.append(masks[b].cpu().numpy())
            if return_predictions:
                pp.append(probs[b].cpu().numpy().transpose(1, 2, 0))
    avg_loss = rl / len(loader.dataset)
    metrics  = compute_metrics(pm, gm)
    if return_predictions:
        return avg_loss, metrics, pm, gm, pp
    return avg_loss, metrics


def train_model(model, train_loader, val_loader, test_loader, criterion,
                optimizer, scheduler, num_epochs=NUM_EPOCHS,
                patience=EARLY_STOP_PATIENCE, model_name='model', device=DEVICE):
    best_miou  = 0.; best_epoch = 0; best_state = None; no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'test_loss': [],
               'train_miou': [], 'val_miou': [],
               'train_pixel_acc': [], 'val_pixel_acc': []}

    print(f"{'='*60}\n  Training: {model_name}\n{'='*60}")
    epoch_pbar = tqdm(range(1, num_epochs + 1), desc='Epochs', unit='epoch')
    for epoch in epoch_pbar:
        t0 = time.time()
        tl     = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = evaluate(model, val_loader,   criterion, device)
        _,  tm = evaluate(model, train_loader, criterion, device)
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(vl)
            else:
                scheduler.step()
        history['train_loss'].append(tl);      history['val_loss'].append(vl)
        history['train_miou'].append(tm['mean_iou'])
        history['val_miou'].append(vm['mean_iou'])
        history['train_pixel_acc'].append(tm['pixel_accuracy'])
        history['val_pixel_acc'].append(vm['pixel_accuracy'])
        elapsed = time.time() - t0
        if vm['mean_iou'] > best_miou:
            best_miou  = vm['mean_iou']; best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0; marker = '  BEST'
        else:
            no_improve += 1; marker = ''
        epoch_pbar.set_postfix({
            'TrLoss': f'{tl:.4f}', 'VaLoss': f'{vl:.4f}',
            'TrmIoU': f'{tm["mean_iou"]:.4f}', 'VamIoU': f'{vm["mean_iou"]:.4f}',
            'Best'  : f'{best_miou:.4f}',
        })
        tqdm.write(
            f'  Epoch {epoch:3d}/{num_epochs} | '
            f'TrLoss:{tl:.4f}  VaLoss:{vl:.4f} | '
            f'TrmIoU:{tm["mean_iou"]:.4f}  VamIoU:{vm["mean_iou"]:.4f} | '
            f'{elapsed:.1f}s{marker}'
        )
        if no_improve >= patience:
            tqdm.write(f'\n  Early stopping at epoch {epoch}'); break

    if best_state:
        model.load_state_dict(best_state)
    sp = OUTPUT_DIR / f'{model_name}_best.pth'
    torch.save(best_state, sp)
    tqdm.write(f'\n  Best Val mIoU : {best_miou:.4f} at epoch {best_epoch}')
    tqdm.write(f'  Model saved   -> {sp}')
    tqdm.write('  Evaluating TEST SET (first and only time)...')
    test_loss, test_metrics = evaluate(model, test_loader, criterion, device)
    history['test_loss'].append(test_loss)
    tqdm.write(f'  Test mIoU : {test_metrics["mean_iou"]:.4f} | '
               f'PixAcc : {test_metrics["pixel_accuracy"]:.4f}')
    return model, history, test_metrics


print('Training loops defined.')

In [ ]:
def plot_training_history(history, model_name='Model'):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    for ax, (tk, vk, yl, ttl) in zip(axes, [
        ('train_loss', 'val_loss', 'Loss', 'Loss'),
        ('train_miou', 'val_miou', 'Mean IoU', 'Mean IoU'),
        ('train_pixel_acc', 'val_pixel_acc', 'Pixel Acc', 'Pixel Accuracy'),
    ]):
        ax.plot(epochs, history[tk], 'b-', label='Train')
        ax.plot(epochs, history[vk], 'r-', label='Val')
        ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
        ax.set_title(f'{model_name} — {ttl}'); ax.legend()
    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def visualize_predictions(model, dataset, indices, model_name='Model',
                          device=DEVICE, num_samples=8):
    model.eval()
    n = min(num_samples, len(indices))
    fig, axes = plt.subplots(n, 3, figsize=(18, 5 * n))
    if n == 1: axes = axes[np.newaxis, :]
    dm = np.array([0.485, 0.456, 0.406]); ds = np.array([0.229, 0.224, 0.225])
    for row, idx in enumerate(indices[:n]):
        it, mt = dataset[idx]
        img_np = (it.numpy().transpose(1, 2, 0) * ds + dm).clip(0, 1)
        with torch.no_grad():
            pm = model(it.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
        gm = mt.numpy()
        gc = np.zeros((*gm.shape, 3), dtype=np.uint8)
        pc = np.zeros((*pm.shape, 3), dtype=np.uint8)
        for c in range(NUM_CLASSES):
            gc[gm == c] = CLASS_COLORS[c]; pc[pm == c] = CLASS_COLORS[c]
        for col, (im, ttl) in enumerate([
            (img_np, 'Input'), (gc, 'Ground Truth'), (pc, 'Prediction')
        ]):
            axes[row, col].imshow(im); axes[row, col].set_title(ttl); axes[row, col].axis('off')
    patches = [mpatches.Patch(color=np.array(CLASS_COLORS[i]) / 255., label=CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    plt.suptitle(f'{model_name} — Predictions', fontsize=14, y=1.01)
    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix(cm, title='Confusion Matrix'):
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=13); plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = cm_norm[i, j]
            if v > 0.005:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color='white' if v > 0.5 else 'black', fontsize=8)
    plt.tight_layout(); return fig


class SemanticSegmentationTarget:
    def __init__(self, category, mask=None): self.category = category; self.mask = mask
    def __call__(self, model_output):
        if self.mask is not None: return (model_output[self.category] * self.mask).sum()
        return model_output[self.category].sum()


def generate_gradcam_for_image(model, img_tensor, target_class, target_layer, device=DEVICE):
    targets = [SemanticSegmentationTarget(target_class)]
    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        gc = cam(input_tensor=img_tensor.unsqueeze(0).to(device), targets=targets)[0]
    dm = np.array([0.485, 0.456, 0.406]); ds = np.array([0.229, 0.224, 0.225])
    img_np = (img_tensor.numpy().transpose(1, 2, 0) * ds + dm).clip(0, 1).astype(np.float32)
    return show_cam_on_image(img_np, gc, use_rgb=True), gc


def visualize_gradcam_gallery(model, dataset, indices, target_classes,
                              target_layer, model_name='Model', device=DEVICE):
    n_imgs = len(indices); n_cls = len(target_classes)
    fig, axes = plt.subplots(n_imgs, n_cls + 1, figsize=(5 * (n_cls + 1), 5 * n_imgs))
    if n_imgs == 1: axes = axes[np.newaxis, :]
    dm = np.array([0.485, 0.456, 0.406]); ds = np.array([0.229, 0.224, 0.225])
    for row, idx in enumerate(indices):
        it, _ = dataset[idx]
        img_np = (it.numpy().transpose(1, 2, 0) * ds + dm).clip(0, 1)
        axes[row, 0].imshow(img_np); axes[row, 0].set_title('Input', fontsize=10)
        axes[row, 0].axis('off')
        for col, ci in enumerate(target_classes):
            cam_img, _ = generate_gradcam_for_image(model, it, ci, target_layer, device)
            axes[row, col + 1].imshow(cam_img)
            axes[row, col + 1].set_title(f'GradCAM: {CLASS_NAMES[ci]}', fontsize=10)
            axes[row, col + 1].axis('off')
    plt.suptitle(f'{model_name} — Grad-CAM', fontsize=14, y=1.01); plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Visualisation and Grad-CAM utilities defined.')

## 6. Visualise Synthetic Background Compositing

In [ ]:
sample_vis = random.sample(train_pairs, 4)
fig, axes  = plt.subplots(2, 4, figsize=(22, 11))

for i, (img_p, ann_p) in enumerate(sample_vis):
    img     = cv2.imread(str(img_p))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]
    fg      = get_foreground_binary_mask(ann_p, h, w)
    fg3     = np.stack([fg] * 3, axis=-1)

    bg_path = synthetic_bg_paths[i * 20]
    bg_img  = cv2.imread(bg_path)
    bg_img  = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
    bg_img  = cv2.resize(bg_img, (w, h))
    composite = np.where(fg3 == 1, img_rgb, bg_img)

    disp = (400, 400)
    axes[0, i].imshow(cv2.resize(img_rgb, disp))
    axes[0, i].set_title(f'Original\n{img_p.name}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(cv2.resize(composite, disp))
    axes[1, i].set_title(f'Synthetic BG: {Path(bg_path).name}', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Step 3 — Synthetic Background Compositing', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'step3_synthetic_compositing_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step3_synthetic_compositing_samples.png')

## 7. Build Synthetic-Background DataLoaders

In [ ]:
print('Building synthetic-BG loaders...')
(train_loader_synth, val_loader_synth, test_loader_synth,
 train_ds_synth,     val_ds_synth,     test_ds_synth) = make_loaders(
    train_pairs, val_pairs, test_pairs,
    bg_mode='synthetic', bg_images=synthetic_bg_paths
)

# Also need natural test loader for cross-BG robustness (scenario b)
print('Building natural test loader for cross-BG evaluation...')
test_ds_natural  = TomatoSegDataset(test_pairs, transform=get_test_augmentation(),
                                    bg_mode='natural')
test_loader_natural = DataLoader(test_ds_natural, batch_size=BATCH_SIZE_TEST,
                                 shuffle=False, num_workers=NUM_WORKERS,
                                 pin_memory=PIN_MEMORY)

print(f'Synthetic — Train: {len(train_ds_synth)} | Val: {len(val_ds_synth)} | Test: {len(test_ds_synth)}')
imgs, masks = next(iter(train_loader_synth))
print(f'Batch — imgs: {tuple(imgs.shape)}  masks: {tuple(masks.shape)}  dtype: {masks.dtype}')

## 8. Train U-Net / MobileNetV2 (Synthetic BG)

In [ ]:
print('Creating U-Net with MobileNetV2 encoder...')
model_mobilenet_synth = create_unet_model('mobilenet_v2')
total     = sum(p.numel() for p in model_mobilenet_synth.parameters())
trainable = sum(p.numel() for p in model_mobilenet_synth.parameters() if p.requires_grad)
print(f'Parameters — Total: {total:,}  Trainable: {trainable:,}')

criterion_m = CombinedLoss(class_weights=class_weights_tensor).to(DEVICE)
optimizer_m = torch.optim.Adam(model_mobilenet_synth.parameters(),
                               lr=LEARNING_RATE, weight_decay=1e-5)
scheduler_m = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_m, T_max=NUM_EPOCHS, eta_min=1e-6)

model_mobilenet_synth, history_m_synth, test_metrics_m_synth = train_model(
    model_mobilenet_synth,
    train_loader_synth, val_loader_synth, test_loader_synth,
    criterion_m, optimizer_m, scheduler_m,
    num_epochs=NUM_EPOCHS, patience=EARLY_STOP_PATIENCE,
    model_name='step3_mobilenetv2_synthetic'
)

## 9. Train U-Net / EfficientNet-B0 (Synthetic BG)

In [ ]:
print('Creating U-Net with EfficientNet-B0 encoder...')
model_effnet_synth = create_unet_model('efficientnet-b0')
total     = sum(p.numel() for p in model_effnet_synth.parameters())
trainable = sum(p.numel() for p in model_effnet_synth.parameters() if p.requires_grad)
print(f'Parameters — Total: {total:,}  Trainable: {trainable:,}')

criterion_e = CombinedLoss(class_weights=class_weights_tensor).to(DEVICE)
optimizer_e = torch.optim.Adam(model_effnet_synth.parameters(),
                               lr=LEARNING_RATE, weight_decay=1e-5)
scheduler_e = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_e, T_max=NUM_EPOCHS, eta_min=1e-6)

model_effnet_synth, history_e_synth, test_metrics_e_synth = train_model(
    model_effnet_synth,
    train_loader_synth, val_loader_synth, test_loader_synth,
    criterion_e, optimizer_e, scheduler_e,
    num_epochs=NUM_EPOCHS, patience=EARLY_STOP_PATIENCE,
    model_name='step3_efficientnetb0_synthetic'
)

## 10. Training Curves

In [ ]:
plot_training_history(history_m_synth, 'Step3 — MobileNetV2 (Synthetic BG)')
plot_training_history(history_e_synth, 'Step3 — EfficientNet-B0 (Synthetic BG)')

## 11. Full Test-Set Evaluation (Matched: Synthetic → Synthetic)

In [ ]:
print('Evaluating MobileNetV2 (synthetic BG)...')
_, metrics_m_synth, pred_masks_m, gt_masks_m, pred_probs_m = evaluate(
    model_mobilenet_synth, test_loader_synth, criterion_m, return_predictions=True)
print_metrics(metrics_m_synth, 'Step 3 — MobileNetV2 (Synthetic BG)')

fig = plot_confusion_matrix(metrics_m_synth['confusion_matrix'],
                            'Step 3 — MobileNetV2 (Synthetic BG)')
plt.savefig(OUTPUT_DIR / 'step3_mobilenetv2_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEvaluating EfficientNet-B0 (synthetic BG)...')
_, metrics_e_synth, pred_masks_e, gt_masks_e, pred_probs_e = evaluate(
    model_effnet_synth, test_loader_synth, criterion_e, return_predictions=True)
print_metrics(metrics_e_synth, 'Step 3 — EfficientNet-B0 (Synthetic BG)')

fig = plot_confusion_matrix(metrics_e_synth['confusion_matrix'],
                            'Step 3 — EfficientNet-B0 (Synthetic BG)')
plt.savefig(OUTPUT_DIR / 'step3_efficientnetb0_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Calibration Analysis — Step 3

In [ ]:
cal_m_synth = compute_calibration(pred_probs_m, gt_masks_m)
cal_e_synth = compute_calibration(pred_probs_e, gt_masks_e)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
plot_reliability_diagram(cal_m_synth, 'MobileNetV2 (Synthetic BG)',     ax=axes[0])
plot_reliability_diagram(cal_e_synth, 'EfficientNet-B0 (Synthetic BG)', ax=axes[1])
plt.suptitle('Step 3 — Calibration: Reliability Diagrams', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'step3_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'ECE — MobileNetV2    : {cal_m_synth["ece"]:.4f}')
print(f'ECE — EfficientNet-B0: {cal_e_synth["ece"]:.4f}')

## 13. Prediction Visualisations

In [ ]:
test_indices = list(range(0, min(len(test_ds_synth), 80), 10))[:8]

visualize_predictions(model_mobilenet_synth, test_ds_synth, test_indices,
                      model_name='Step3 — MobileNetV2 (Synthetic BG)')
visualize_predictions(model_effnet_synth, test_ds_synth, test_indices,
                      model_name='Step3 — EfficientNet-B0 (Synthetic BG)')

## 14. Grad-CAM Explainability

In [ ]:
gradcam_classes = [0, 1, 3, 4, 6]
gradcam_indices = list(range(0, min(len(test_ds_synth), 50), 10))[:5]

target_layer_m = model_mobilenet_synth.segmentation_head[0]
target_layer_e = model_effnet_synth.segmentation_head[0]

print('Generating Grad-CAM for MobileNetV2 (Synthetic BG)...')
visualize_gradcam_gallery(
    model_mobilenet_synth, test_ds_synth, gradcam_indices,
    gradcam_classes, target_layer_m, model_name='Step3 — MobileNetV2 (Synthetic BG)')

print('\nGenerating Grad-CAM for EfficientNet-B0 (Synthetic BG)...')
visualize_gradcam_gallery(
    model_effnet_synth, test_ds_synth, gradcam_indices,
    gradcam_classes, target_layer_e, model_name='Step3 — EfficientNet-B0 (Synthetic BG)')

## 15. Cross-Background Robustness Evaluation

Three transfer scenarios to measure background bias:

| Scenario | Train BG | Test BG | What it measures |
|----------|----------|---------|-----------------|
| (a) | Natural | Synthetic | Does natural-BG training generalise to unseen synthetic BG? |
| (b) | Synthetic | Natural | Does synthetic-BG training generalise back to real images? |
| (c) | Synthetic | Synthetic | Matched upper bound for synthetic-trained models |

A large gap between (c) and (b) indicates the model has overfit to synthetic backgrounds.
A large gap between baseline Step 1 and scenario (a) shows natural BG dependency.

In [ ]:
# ── Load Step 1 (natural-trained) model checkpoints for scenario (a) ─────────
# We reload the saved weights so we can cross-evaluate without re-training.

print('Loading Step 1 model checkpoints for cross-BG evaluation...')

# Dummy criterion for evaluate() signature — weights already applied during training
dummy_criterion = CombinedLoss(class_weights=class_weights_tensor).to(DEVICE)

model_m_nat_reload = create_unet_model('mobilenet_v2')
ckpt_m_nat = OUTPUT_DIR / 'step1_mobilenetv2_natural_best.pth'
assert ckpt_m_nat.exists(), f'Missing checkpoint: {ckpt_m_nat}. Run notebook 04 first.'
model_m_nat_reload.load_state_dict(torch.load(ckpt_m_nat, map_location=DEVICE))
model_m_nat_reload.eval()
print(f'  Loaded: {ckpt_m_nat.name}')

model_e_nat_reload = create_unet_model('efficientnet-b0')
ckpt_e_nat = OUTPUT_DIR / 'step1_efficientnetb0_natural_best.pth'
assert ckpt_e_nat.exists(), f'Missing checkpoint: {ckpt_e_nat}. Run notebook 04 first.'
model_e_nat_reload.load_state_dict(torch.load(ckpt_e_nat, map_location=DEVICE))
model_e_nat_reload.eval()
print(f'  Loaded: {ckpt_e_nat.name}')

print('Checkpoints loaded.')

In [ ]:
# ── Scenario (a): Natural-trained → Synthetic test ───────────────────────────
print('\n--- Scenario (a): Natural-trained -> Synthetic test ---')

_, rob_m_nat2synth = evaluate(model_m_nat_reload, test_loader_synth, dummy_criterion)
print_metrics(rob_m_nat2synth, 'Scenario (a): MobileNetV2 | Trained: Natural | Tested: Synthetic')

_, rob_e_nat2synth = evaluate(model_e_nat_reload, test_loader_synth, dummy_criterion)
print_metrics(rob_e_nat2synth, 'Scenario (a): EfficientNet-B0 | Trained: Natural | Tested: Synthetic')

In [ ]:
# ── Scenario (b): Synthetic-trained → Natural test ───────────────────────────
print('\n--- Scenario (b): Synthetic-trained -> Natural test ---')

_, rob_m_synth2nat = evaluate(model_mobilenet_synth, test_loader_natural, criterion_m)
print_metrics(rob_m_synth2nat, 'Scenario (b): MobileNetV2 | Trained: Synthetic | Tested: Natural')

_, rob_e_synth2nat = evaluate(model_effnet_synth, test_loader_natural, criterion_e)
print_metrics(rob_e_synth2nat, 'Scenario (b): EfficientNet-B0 | Trained: Synthetic | Tested: Natural')

In [ ]:
# ── Scenario (c): Synthetic-trained → Synthetic test (matched — already done) ─
print('\n--- Scenario (c): Synthetic-trained -> Synthetic test (matched) ---')
print('Already computed in Section 11:')
print(f'  MobileNetV2    mIoU: {metrics_m_synth["mean_iou"]:.4f}')
print(f'  EfficientNet-B0 mIoU: {metrics_e_synth["mean_iou"]:.4f}')

In [ ]:
# ── Robustness summary table ──────────────────────────────────────────────────
rob_rows = []
for enc, s1_mat, s1_nat, a_mat, b_mat, c_mat in [
    ('MobileNetV2',
     metrics_m_natural,   # Step 1 baseline (Natural → Natural)
     metrics_m_natural,
     rob_m_nat2synth,     # (a) Natural → Synthetic
     rob_m_synth2nat,     # (b) Synthetic → Natural
     metrics_m_synth),    # (c) Synthetic → Synthetic
    ('EfficientNet-B0',
     metrics_e_natural,
     metrics_e_natural,
     rob_e_nat2synth,
     rob_e_synth2nat,
     metrics_e_synth),
]:
    rob_rows.append({'Encoder': enc, 'Scenario': 'Baseline: Natural→Natural',
                     'Train BG': 'Natural', 'Test BG': 'Natural',
                     'mIoU': f"{s1_nat['mean_iou']:.4f}",
                     'Pixel Acc': f"{s1_nat['pixel_accuracy']:.4f}"})
    rob_rows.append({'Encoder': enc, 'Scenario': '(a) Natural→Synthetic',
                     'Train BG': 'Natural', 'Test BG': 'Synthetic',
                     'mIoU': f"{a_mat['mean_iou']:.4f}",
                     'Pixel Acc': f"{a_mat['pixel_accuracy']:.4f}"})
    rob_rows.append({'Encoder': enc, 'Scenario': '(b) Synthetic→Natural',
                     'Train BG': 'Synthetic', 'Test BG': 'Natural',
                     'mIoU': f"{b_mat['mean_iou']:.4f}",
                     'Pixel Acc': f"{b_mat['pixel_accuracy']:.4f}"})
    rob_rows.append({'Encoder': enc, 'Scenario': '(c) Synthetic→Synthetic',
                     'Train BG': 'Synthetic', 'Test BG': 'Synthetic',
                     'mIoU': f"{c_mat['mean_iou']:.4f}",
                     'Pixel Acc': f"{c_mat['pixel_accuracy']:.4f}"})

rob_df = pd.DataFrame(rob_rows)
print('\n=== Cross-Background Robustness Summary ===')
print(rob_df.to_string(index=False))

In [ ]:
# ── Robustness bar chart ─────────────────────────────────────────────────────
scenarios  = ['Baseline\nNat→Nat', '(a) Nat→Syn', '(b) Syn→Nat', '(c) Syn→Syn']
miou_m = [metrics_m_natural['mean_iou'], rob_m_nat2synth['mean_iou'],
           rob_m_synth2nat['mean_iou'],  metrics_m_synth['mean_iou']]
miou_e = [metrics_e_natural['mean_iou'], rob_e_nat2synth['mean_iou'],
           rob_e_synth2nat['mean_iou'],  metrics_e_synth['mean_iou']]

x     = np.arange(len(scenarios))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 6))
bars_m = ax.bar(x - width / 2, miou_m, width, label='MobileNetV2',    color='#2196F3', alpha=0.85)
bars_e = ax.bar(x + width / 2, miou_e, width, label='EfficientNet-B0', color='#FF5722', alpha=0.85)

for bars in [bars_m, bars_e]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylabel('Mean IoU', fontsize=12); ax.set_ylim(0, 1.05)
ax.set_title('Cross-Background Robustness — mIoU Across Transfer Scenarios',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.axhline(0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'step3_robustness_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step3_robustness_bar_chart.png')

## 16. Steps 1–2–3 Comparison

Full comparison across all three background conditions and both encoders.

In [ ]:
# ── Metrics table: all steps × both encoders ─────────────────────────────────
all_rows = []
conditions = [
    ('Step 1: Natural',   metrics_m_natural, metrics_e_natural, cal_m_natural, cal_e_natural),
    ('Step 2: Removed',   metrics_m_removed, metrics_e_removed, cal_m_removed, cal_e_removed),
    ('Step 3: Synthetic', metrics_m_synth,   metrics_e_synth,   cal_m_synth,   cal_e_synth),
]
for cond, mm, me, cm_cal, ce_cal in conditions:
    for enc, m, cal in [('MobileNetV2', mm, cm_cal), ('EfficientNet-B0', me, ce_cal)]:
        all_rows.append({
            'Condition' : cond,
            'Encoder'   : enc,
            'Pixel Acc' : f"{m['pixel_accuracy']:.4f}",
            'Mean IoU'  : f"{m['mean_iou']:.4f}",
            'Mean Dice' : f"{m['mean_dice']:.4f}",
            'ECE'       : f"{cal['ece']:.4f}",
        })

all_df = pd.DataFrame(all_rows)
print('\n=== Steps 1–2–3 Full Metrics Comparison ===')
print(all_df.to_string(index=False))

In [ ]:
# ── mIoU grouped bar chart: all 3 steps × 2 encoders ─────────────────────────
step_labels = ['Step 1\nNatural', 'Step 2\nRemoved', 'Step 3\nSynthetic']
miou_mob = [metrics_m_natural['mean_iou'], metrics_m_removed['mean_iou'], metrics_m_synth['mean_iou']]
miou_eff = [metrics_e_natural['mean_iou'], metrics_e_removed['mean_iou'], metrics_e_synth['mean_iou']]

x = np.arange(3); width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: mIoU
bars_m = axes[0].bar(x - width/2, miou_mob, width, label='MobileNetV2',    color='#2196F3', alpha=0.85)
bars_e = axes[0].bar(x + width/2, miou_eff, width, label='EfficientNet-B0', color='#FF5722', alpha=0.85)
for bars in [bars_m, bars_e]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.005,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(step_labels, fontsize=11)
axes[0].set_ylabel('Mean IoU', fontsize=11); axes[0].set_ylim(0, 1.0)
axes[0].set_title('Mean IoU — Steps 1 vs 2 vs 3', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: ECE
ece_mob = [cal_m_natural['ece'], cal_m_removed['ece'], cal_m_synth['ece']]
ece_eff = [cal_e_natural['ece'], cal_e_removed['ece'], cal_e_synth['ece']]
bars_m2 = axes[1].bar(x - width/2, ece_mob, width, label='MobileNetV2',    color='#2196F3', alpha=0.85)
bars_e2 = axes[1].bar(x + width/2, ece_eff, width, label='EfficientNet-B0', color='#FF5722', alpha=0.85)
for bars in [bars_m2, bars_e2]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.001,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(step_labels, fontsize=11)
axes[1].set_ylabel('ECE (lower = better calibrated)', fontsize=11); axes[1].set_ylim(0, 0.3)
axes[1].set_title('ECE — Steps 1 vs 2 vs 3', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Steps 1–2–3: Mean IoU and Calibration Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'steps123_miou_ece_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: steps123_miou_ece_comparison.png')

In [ ]:
# ── Per-class IoU heatmap: 6 conditions × 7 classes ─────────────────────────
# Rows: 6 model-condition combinations
# Cols: 7 classes

row_labels = [
    'MobileNetV2 — Natural',
    'MobileNetV2 — Removed',
    'MobileNetV2 — Synthetic',
    'EfficientNet-B0 — Natural',
    'EfficientNet-B0 — Removed',
    'EfficientNet-B0 — Synthetic',
]

iou_matrix = np.array([
    np.nan_to_num(metrics_m_natural['iou_per_class'], nan=0.),
    np.nan_to_num(metrics_m_removed['iou_per_class'], nan=0.),
    np.nan_to_num(metrics_m_synth  ['iou_per_class'], nan=0.),
    np.nan_to_num(metrics_e_natural['iou_per_class'], nan=0.),
    np.nan_to_num(metrics_e_removed['iou_per_class'], nan=0.),
    np.nan_to_num(metrics_e_synth  ['iou_per_class'], nan=0.),
])   # shape (6, 7)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(iou_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='IoU')

ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=10)
ax.set_yticks(range(6))
ax.set_yticklabels(row_labels, fontsize=10)

# Add value annotations
for i in range(6):
    for j in range(NUM_CLASSES):
        v = iou_matrix[i, j]
        text_color = 'black' if 0.3 < v < 0.8 else 'white'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                color=text_color, fontweight='bold')

# Horizontal separator between encoders
ax.axhline(2.5, color='white', linewidth=3)

ax.set_title('Per-Class IoU Heatmap — All Steps × Both Encoders',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'steps123_per_class_iou_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: steps123_per_class_iou_heatmap.png')

## 17. Save Results for Grand Comparison

In [ ]:
# ── Step 3 results ───────────────────────────────────────────────────────────
step3_results = {
    'mobilenet'   : {'metrics': metrics_m_synth, 'calibration': cal_m_synth,
                     'history': history_m_synth},
    'efficientnet': {'metrics': metrics_e_synth, 'calibration': cal_e_synth,
                     'history': history_e_synth},
}
with open(OUTPUT_DIR / 'step3_results.pkl', 'wb') as f:
    pickle.dump(step3_results, f)
print('Saved -> step3_results.pkl')

# ── Robustness results ────────────────────────────────────────────────────────
robustness_results = {
    'mobilenet': {
        'nat_to_nat'   : metrics_m_natural,   # Baseline Step 1
        'nat_to_synth' : rob_m_nat2synth,     # Scenario (a)
        'synth_to_nat' : rob_m_synth2nat,     # Scenario (b)
        'synth_to_synth': metrics_m_synth,    # Scenario (c)
    },
    'efficientnet': {
        'nat_to_nat'   : metrics_e_natural,
        'nat_to_synth' : rob_e_nat2synth,
        'synth_to_nat' : rob_e_synth2nat,
        'synth_to_synth': metrics_e_synth,
    },
}
with open(OUTPUT_DIR / 'robustness_results.pkl', 'wb') as f:
    pickle.dump(robustness_results, f)
print('Saved -> robustness_results.pkl')

## 18. Summary

In [ ]:
print('=' * 70)
print('  NOTEBOOK 06 — STEP 3 COMPLETE')
print('=' * 70)
print('  Background condition : Synthetic (200 procedural backgrounds)')
print()
print(f'  {"Model":<22} {"Condition":<18} {"Mean IoU":>10} {"ECE":>8}')
print(f'  {"-"*60}')
for enc, m_nat, m_rem, m_syn, cal_nat, cal_rem, cal_syn in [
    ('MobileNetV2',
     metrics_m_natural, metrics_m_removed, metrics_m_synth,
     cal_m_natural, cal_m_removed, cal_m_synth),
    ('EfficientNet-B0',
     metrics_e_natural, metrics_e_removed, metrics_e_synth,
     cal_e_natural, cal_e_removed, cal_e_synth),
]:
    print(f'  {enc:<22} {"Natural BG":<18} {m_nat["mean_iou"]:>10.4f} {cal_nat["ece"]:>8.4f}')
    print(f'  {"" :<22} {"BG Removed":<18} {m_rem["mean_iou"]:>10.4f} {cal_rem["ece"]:>8.4f}')
    print(f'  {"" :<22} {"Synthetic BG":<18} {m_syn["mean_iou"]:>10.4f} {cal_syn["ece"]:>8.4f}')
    print()

print('  Cross-BG Robustness (mIoU):')
print(f'  {"Scenario":<35} {"MobileNetV2":>14} {"EfficientNet-B0":>16}')
print(f'  {"-"*67}')
for label, mm, me in [
    ('Baseline: Natural → Natural',    metrics_m_natural["mean_iou"],  metrics_e_natural["mean_iou"]),
    ('(a) Natural → Synthetic',        rob_m_nat2synth["mean_iou"],    rob_e_nat2synth["mean_iou"]),
    ('(b) Synthetic → Natural',        rob_m_synth2nat["mean_iou"],    rob_e_synth2nat["mean_iou"]),
    ('(c) Synthetic → Synthetic',      metrics_m_synth["mean_iou"],    metrics_e_synth["mean_iou"]),
]:
    print(f'  {label:<35} {mm:>14.4f} {me:>16.4f}')

print()
print('  Saved files:')
for fp in sorted(OUTPUT_DIR.glob('step3_*')) + sorted(OUTPUT_DIR.glob('steps123_*')) + \
          sorted(OUTPUT_DIR.glob('robustness_*')):
    print(f'    {fp.name}')
print()
print('  Next: run 07_grand_comparison.ipynb')
print('=' * 70)

In [1]:
!jupyter nbconvert --to html 06_step3_synthetic_backgrounds.ipynb 

[NbConvertApp] Converting notebook 06_step3_synthetic_backgrounds.ipynb to html
[NbConvertApp] Writing 544198 bytes to 06_step3_synthetic_backgrounds.html
